In [1]:
import pandas as pd 
df = pd.read_csv('C:\\Users\\USER\\Documents\\lexi-codes\\data science\\datasets\\realtor-data.zip.csv\\realtor-data.zip.csv',dtype={'status':'category'})
df.head(6)

,brokered_by,status,price,bed,bath,acre_lot,street,city,state,zip_code,house_size,prev_sold_date
0,103378.0,for_sale,105000.0,3.0,2.0,0.12,1962661.0,Adjuntas,Puerto Rico,601.0,920.0,NaN
1,52707.0,for_sale,80000.0,4.0,2.0,0.08,1902874.0,Adjuntas,Puerto Rico,601.0,1527.0,NaN
2,103379.0,for_sale,67000.0,2.0,1.0,0.15,1404990.0,Juana Diaz,Puerto Rico,795.0,748.0,NaN
3,31239.0,for_sale,145000.0,4.0,2.0,0.10,1947675.0,Ponce,Puerto Rico,731.0,1800.0,NaN
4,34632.0,for_sale,65000.0,6.0,2.0,0.05,331151.0,Mayaguez,Puerto Rico,680.0,NaN,NaN
5,103378.0,for_sale,179000.0,4.0,3.0,0.46,1850806.0,San Sebastian,Puerto Rico,612.0,2520.0,NaN


In [2]:
df.dtypes

brokered_by        float64
status            category
price              float64
bed                float64
bath               float64
acre_lot           float64
street             float64
city                object
state               object
zip_code           float64
house_size         float64
prev_sold_date      object
dtype: object

In [3]:
df.count()

brokered_by       2221849
status            2226382
price             2224841
bed               1745065
bath              1714611
acre_lot          1900793
street            2215516
city              2224975
state             2226374
zip_code          2226083
house_size        1657898
prev_sold_date    1492085
dtype: int64

In [4]:
df.count().min()

1492085

In [5]:
df['brokered_by'].nunique()

110143

In [6]:
df['status'].nunique()

3

In [7]:
df['city'].nunique()

20098

In [8]:
df['street'].nunique()

2001358

In [9]:
df['zip_code'].nunique()

30334

In [10]:
clean_df = df.dropna()
clean_df.count()

brokered_by       1084909
status            1084909
price             1084909
bed               1084909
bath              1084909
acre_lot          1084909
street            1084909
city              1084909
state             1084909
zip_code          1084909
house_size        1084909
prev_sold_date    1084909
dtype: int64

TOTAL NUMBER OF ROWS WITH AT LEAST ONE EMPTY COLUMN VALUE.

In [11]:
2226382-1084909

1141473

DATAFRAME CONTAINING ONLY ROWS WITH CRITICAL COLUMN(PRICE), COMPLETE.

In [12]:
df_price_complete = df.dropna(subset=['price'])
print(f"Original rows: {len(df)}")
print(f"Rows with price: {len(df_price_complete)}")
print(f"Rows dropped: {len(df) - len(df_price_complete)}")
print(f"Percentage kept: {len(df_price_complete)/len(df)*100:.2f}%")

Original rows: 2226382
Rows with price: 2224841
Rows dropped: 1541
Percentage kept: 99.93%


WORKING DATAFRAME FOR FURTHER CLEANING

In [13]:
working_df = df.dropna(subset=['brokered_by', 'price', 'street', 'city', 'state','zip_code'])
print(f"Original rows: {len(df)}")
print(f"Rows with clean working df: {len(working_df)}")
print(f"Rows dropped: {len(df) - len(working_df)}")
print(f"Percentage kept: {len(working_df)/len(df)*100:.2f}%")

Original rows: 2226382
Rows with clean working df: 2207981
Rows dropped: 18401
Percentage kept: 99.17%


In [14]:
working_df.count()

brokered_by       2207981
status            2207981
price             2207981
bed               1733776
bath              1704415
acre_lot          1886350
street            2207981
city              2207981
state             2207981
zip_code          2207981
house_size        1647174
prev_sold_date    1482400
dtype: int64

In [15]:
working_df = working_df.copy() 
working_df['prev_sold_date'] = working_df['prev_sold_date'].fillna('Not Previously Sold')

In [16]:
working_df.head(3)

,brokered_by,status,price,bed,bath,acre_lot,street,city,state,zip_code,house_size,prev_sold_date
0,103378.0,for_sale,105000.0,3.0,2.0,0.12,1962661.0,Adjuntas,Puerto Rico,601.0,920.0,Not Previously Sold
1,52707.0,for_sale,80000.0,4.0,2.0,0.08,1902874.0,Adjuntas,Puerto Rico,601.0,1527.0,Not Previously Sold
2,103379.0,for_sale,67000.0,2.0,1.0,0.15,1404990.0,Juana Diaz,Puerto Rico,795.0,748.0,Not Previously Sold


CLEANING REMAINING NAN VALUES WITH MEDIAN WITH NO GROUPING APPLIED

In [17]:
working_df_no_group_clean = working_df.copy()
for column in ['bed', 'bath', 'acre_lot', 'house_size']:
    median_value = working_df_no_group_clean[column].median()
    working_df_no_group_clean[column] = working_df_no_group_clean[column].fillna(median_value)
    print(f"Filled {column} NaN with median: {median_value}")

Filled bed NaN with median: 3.0
Filled bath NaN with median: 2.0
Filled acre_lot NaN with median: 0.26
Filled house_size NaN with median: 1760.0


In [18]:
working_df_no_group_clean.count()

brokered_by       2207981
status            2207981
price             2207981
bed               2207981
bath              2207981
acre_lot          2207981
street            2207981
city              2207981
state             2207981
zip_code          2207981
house_size        2207981
prev_sold_date    2207981
dtype: int64

CLEANING REMAINING NAN VALUES WITH MEDIAN WITH GROUPING BY CITY APPLIED

each data is group into dataframes by cities they fall into and any NAN value is filled with that groups median num

In [19]:
working_df_group_clean = working_df.copy()
grouping = working_df_group_clean.groupby('city')
for col in ['bed', 'bath', 'house_size','acre_lot']:
    city_medians = working_df_group_clean.groupby('city')[col].median()
    
    working_df_group_clean[col] = working_df_group_clean[col].fillna(working_df_group_clean['city'].map(city_medians))
    
    print(f"Filled {col} with city-specific medians")

Filled bed with city-specific medians
Filled bath with city-specific medians
Filled house_size with city-specific medians
Filled acre_lot with city-specific medians


In [20]:
grouping.count()

,brokered_by,status,price,bed,bath,acre_lot,street,state,zip_code,house_size,prev_sold_date
city,,,,,,,,,,,
100 89 Lower Shepard Creek Road,1,1,1,1,1,1,1,1,1,0,1
139th Ave Unit Peck,1,1,1,0,0,1,1,1,1,0,1
15th Ave Milton,1,1,1,0,0,1,1,1,1,0,1
177th Ave Wabasha,1,1,1,0,0,1,1,1,1,0,1
178th Ave Wabasha,1,1,1,0,0,1,1,1,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...
Zumbro Falls,13,13,13,13,13,13,13,13,13,13,13
Zumbrota,34,34,34,34,34,34,34,34,34,34,34
Zuni,2,2,2,2,2,2,2,2,2,2,2


In [21]:
city_medians.nunique()

3311

In [22]:
working_df_group_clean.count()

brokered_by       2207981
status            2207981
price             2207981
bed               2203370
bath              2202793
acre_lot          2206824
street            2207981
city              2207981
state             2207981
zip_code          2207981
house_size        2200874
prev_sold_date    2207981
dtype: int64

In [23]:
working_df['city'].value_counts()

city
Houston                  23849
Chicago                  18059
New York City            11864
Jacksonville             11676
Philadelphia             10428
                         ...  
Mallard                      1
South Taft Mason City        1
Reasnor                      1
Hiteman                      1
Kahlotus                     1
Name: count, Length: 19675, dtype: int64

In [24]:
# Drop all remaining NaN values
working_df_nonan = working_df_group_clean.dropna()

# Check the results
print(f"Original working_df_group_clean rows: {len(working_df_group_clean)}")
print(f"Final rows after dropping: {len(working_df_nonan)}")
print(f"Rows dropped: {len(working_df_group_clean) - len(working_df_nonan)}")
print(f"Percentage kept: {len(working_df_nonan)/len(working_df_group_clean)*100:.2f}%")

# Verify no missing values remain
print(f"\nMissing values check:\n{working_df_nonan.isna().sum()}")

Original working_df_group_clean rows: 2207981
Final rows after dropping: 2199643
Rows dropped: 8338
Percentage kept: 99.62%

Missing values check:
brokered_by       0
status            0
price             0
bed               0
bath              0
acre_lot          0
street            0
city              0
state             0
zip_code          0
house_size        0
prev_sold_date    0
dtype: int64


Total rows drop from start

In [25]:
2226382 - 2199643

26739

In [26]:
# Drop the 'street' column
working_df_final = working_df_nonan.drop(columns=['street'])

# Verify it's gone
print(f"Columns after dropping 'street': {working_df_final.columns.tolist()}")
print(f"\nShape of final dataset: {working_df_final.shape}")
print(f"Rows: {working_df_final.shape[0]:,}, Columns: {working_df_final.shape[1]}")

Columns after dropping 'street': ['brokered_by', 'status', 'price', 'bed', 'bath', 'acre_lot', 'city', 'state', 'zip_code', 'house_size', 'prev_sold_date']

Shape of final dataset: (2199643, 11)
Rows: 2,199,643, Columns: 11


In [27]:
working_df_final.count()

brokered_by       2199643
status            2199643
price             2199643
bed               2199643
bath              2199643
acre_lot          2199643
city              2199643
state             2199643
zip_code          2199643
house_size        2199643
prev_sold_date    2199643
dtype: int64

In [28]:
working_df_final.head(10)

,brokered_by,status,price,bed,bath,acre_lot,city,state,zip_code,house_size,prev_sold_date
0,103378.0,for_sale,105000.0,3.0,2.0,0.12,Adjuntas,Puerto Rico,601.0,920.0,Not Previously Sold
1,52707.0,for_sale,80000.0,4.0,2.0,0.08,Adjuntas,Puerto Rico,601.0,1527.0,Not Previously Sold
2,103379.0,for_sale,67000.0,2.0,1.0,0.15,Juana Diaz,Puerto Rico,795.0,748.0,Not Previously Sold
3,31239.0,for_sale,145000.0,4.0,2.0,0.10,Ponce,Puerto Rico,731.0,1800.0,Not Previously Sold
4,34632.0,for_sale,65000.0,6.0,2.0,0.05,Mayaguez,Puerto Rico,680.0,1550.0,Not Previously Sold
5,103378.0,for_sale,179000.0,4.0,3.0,0.46,San Sebastian,Puerto Rico,612.0,2520.0,Not Previously Sold
6,1205.0,for_sale,50000.0,3.0,1.0,0.20,Ciales,Puerto Rico,639.0,2040.0,Not Previously Sold
7,50739.0,for_sale,71600.0,3.0,2.0,0.08,Ponce,Puerto Rico,731.0,1050.0,Not Previously Sold
8,81909.0,for_sale,100000.0,2.0,1.0,0.09,Ponce,Puerto Rico,730.0,1092.0,Not Previously Sold
9,65672.0,for_sale,300000.0,5.0,3.0,7.46,Las Marias,Puerto Rico,670.0,5403.0,Not Previously Sold


In [29]:
working_df_final['prev_sold_date'].nunique()

14931

In [30]:
working_df_final['prev_sold_date'].value_counts()

prev_sold_date
Not Previously Sold    719703
2022-03-31              16960
2022-04-15              16164
2022-04-22              15627
2022-04-08              14902
                        ...  
1972-07-28                  1
1957-04-17                  1
1981-05-13                  1
2017-06-11                  1
1967-01-12                  1
Name: count, Length: 14931, dtype: int64

In [31]:
working_df_final_no_prev_sold = working_df_final.drop(columns=['prev_sold_date'])

# Check both versions
print("=== DATASET VERSIONS ===")
print(f"Version 1 (with prev_sold_date): {working_df_final.shape}")
print(f"Version 2 (without prev_sold_date): {working_df_final_no_prev_sold.shape}")

print(f"\n=== COLUMNS ===")
print(f"With prev_sold_date: {working_df_final.columns.tolist()}")
print(f"Without prev_sold_date: {working_df_final_no_prev_sold.columns.tolist()}")

# summary
print(f"\n=== DATASET SUMMARY ===")
print(f"Total rows: {len(working_df_final):,}")
print(f"Date range in prev_sold_date: {working_df_final['prev_sold_date'].replace('Not Previously Sold', pd.NA).dropna().min()} to {working_df_final['prev_sold_date'].replace('Not Previously Sold', pd.NA).dropna().max()}")
print(f"Properties with previous sale date: {(working_df_final['prev_sold_date'] != 'Not Previously Sold').sum():,} ({(working_df_final['prev_sold_date'] != 'Not Previously Sold').sum()/len(working_df_final)*100:.1f}%)")

=== DATASET VERSIONS ===
Version 1 (with prev_sold_date): (2199643, 11)
Version 2 (without prev_sold_date): (2199643, 10)

=== COLUMNS ===
With prev_sold_date: ['brokered_by', 'status', 'price', 'bed', 'bath', 'acre_lot', 'city', 'state', 'zip_code', 'house_size', 'prev_sold_date']
Without prev_sold_date: ['brokered_by', 'status', 'price', 'bed', 'bath', 'acre_lot', 'city', 'state', 'zip_code', 'house_size']

=== DATASET SUMMARY ===
Total rows: 2,199,643
Date range in prev_sold_date: 1901-01-01 to 3019-04-02
Properties with previous sale date: 1,479,940 (67.3%)


In [32]:
working_df_final.to_csv('housing_data_with_sold_date.csv', index=False)
working_df_final_no_prev_sold.to_csv('housing_data_without_sold_date.csv', index=False)